# TimesFM: Фундаментальная модель от Google Research

Этот notebook содержит код из главы книги "Нейросети для прогнозирования временных рядов".

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/privettoha/neural-forecast-book/blob/main/notebooks/33_timesfm.ipynb)

## Установка зависимостей

In [ ]:
# TimesFM установка
!pip install -q timesfm torch pandas numpy matplotlib

## Подготовка данных

In [ ]:
import torch
import numpy as np
import pandas as pd

# Настройка для оптимизации производительности
torch.set_float32_matmul_precision("high")

# Создаём тестовые данные
np.random.seed(42)

# Линейный тренд
linear_trend = np.linspace(0, 1, 100)

# Синусоида
sinusoid = np.sin(np.linspace(0, 20, 67))

# Реальные данные с сезонностью
t = np.arange(200)
seasonal_data = 100 + np.cumsum(np.random.randn(200)) + 20 * np.sin(t / 7 * 2 * np.pi)

print(f"Linear trend: {linear_trend.shape}")
print(f"Sinusoid: {sinusoid.shape}")
print(f"Seasonal data: {seasonal_data.shape}")

## TimesFM: загрузка и прогнозирование

In [ ]:
import timesfm

# Загружаем модель TimesFM 2.5
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

# Конфигурируем параметры инференса
model.compile(
    timesfm.ForecastConfig(
        max_context=1024,      # максимальная длина контекста
        max_horizon=256,       # максимальный горизонт прогноза
        normalize_inputs=True, # нормализация входов
        use_continuous_quantile_head=True,  # квантильные прогнозы
        force_flip_invariance=True,         # инвариантность к знаку
        infer_is_positive=True,             # неотрицательные значения
        fix_quantile_crossing=True,         # коррекция квантилей
    )
)

print("Модель загружена!")

In [ ]:
# Генерируем прогноз для тестовых данных
point_forecast, quantile_forecast = model.forecast(
    horizon=12,
    inputs=[
        linear_trend,
        sinusoid,
    ],
)

print(f"Форма точечного прогноза: {point_forecast.shape}")      # (2, 12)
print(f"Форма квантильного прогноза: {quantile_forecast.shape}") # (2, 12, 10)

## Визуализация прогнозов

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Линейный тренд
ax = axes[0]
ax.plot(range(len(linear_trend)), linear_trend, 'b-', label='История')
forecast_idx = range(len(linear_trend), len(linear_trend) + 12)
ax.plot(forecast_idx, point_forecast[0], 'r-', label='Прогноз')
ax.axvline(x=len(linear_trend), color='gray', linestyle='--', alpha=0.5)
ax.set_title('Линейный тренд')
ax.legend()
ax.grid(True, alpha=0.3)

# Синусоида
ax = axes[1]
ax.plot(range(len(sinusoid)), sinusoid, 'b-', label='История')
forecast_idx = range(len(sinusoid), len(sinusoid) + 12)
ax.plot(forecast_idx, point_forecast[1], 'r-', label='Прогноз')
ax.axvline(x=len(sinusoid), color='gray', linestyle='--', alpha=0.5)
ax.set_title('Синусоида')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('TimesFM: прогнозы для разных паттернов', fontsize=14)
plt.tight_layout()
plt.show()

## Прогноз с квантилями

In [ ]:
# Прогноз для сезонных данных
point_forecast_seasonal, quantile_forecast_seasonal = model.forecast(
    horizon=24,
    inputs=[seasonal_data],
)

fig, ax = plt.subplots(figsize=(12, 5))

# История (последние 100 точек)
history = seasonal_data[-100:]
ax.plot(range(len(history)), history, 'b-', label='История')

# Прогноз
horizon = 24
forecast_start = len(history)
forecast_idx = range(forecast_start, forecast_start + horizon)

# Точечный прогноз
ax.plot(forecast_idx, point_forecast_seasonal[0], 'r-', linewidth=2, label='Прогноз')

# Квантили (интервалы)
# quantile_forecast имеет 10 квантилей: 0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95
q_low = quantile_forecast_seasonal[0, :, 0]   # 0.05 квантиль
q_high = quantile_forecast_seasonal[0, :, -1] # 0.95 квантиль

ax.fill_between(forecast_idx, q_low, q_high, 
                alpha=0.3, color='red', label='90% интервал')

ax.axvline(x=forecast_start, color='gray', linestyle='--', alpha=0.5)
ax.set_title('TimesFM: прогноз с квантилями')
ax.set_xlabel('Время')
ax.set_ylabel('Значение')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Работа с pandas DataFrame

In [ ]:
def timesfm_forecast_df(df, model, horizon, unique_id_col='unique_id', value_col='y'):
    """
    Прогнозирование TimesFM для DataFrame.
    """
    results = []
    
    segments = df[unique_id_col].unique()
    inputs = []
    
    for seg in segments:
        seg_data = df[df[unique_id_col] == seg].sort_values('ds')
        inputs.append(seg_data[value_col].values)
    
    # Прогноз для всех сегментов
    point_forecasts, quantile_forecasts = model.forecast(
        horizon=horizon,
        inputs=inputs,
    )
    
    # Формируем результат
    for i, seg in enumerate(segments):
        for h in range(horizon):
            results.append({
                unique_id_col: seg,
                'horizon': h + 1,
                'forecast': point_forecasts[i, h],
                'p5': quantile_forecasts[i, h, 0],
                'p95': quantile_forecasts[i, h, -1],
            })
    
    return pd.DataFrame(results)

# Создаём тестовый DataFrame
dates = pd.date_range('2023-01-01', periods=200, freq='D')
test_df = pd.DataFrame({
    'unique_id': np.repeat(['A', 'B'], 200),
    'ds': np.tile(dates, 2),
    'y': np.random.randn(400).cumsum() + 100
})

# Прогноз
forecast_df = timesfm_forecast_df(test_df, model, horizon=16)
print(forecast_df.head(20))